In [0]:
dbutils.widgets.text("catalog", "anurag_dev")
dbutils.widgets.text("src_schema", "silver")
dbutils.widgets.text("tgt_schema", "gold")

catalog = dbutils.widgets.get("catalog")
src = dbutils.widgets.get("src_schema")
tgt = dbutils.widgets.get("tgt_schema")

from pyspark.sql.functions import sum, count, avg, col, month, year

# ---- fact_sales ----
orders = spark.table(f"{catalog}.{src}.orders_enriched") \
    .select("order_id", "customer_id", "order_date", "order_status", "total_amount", "name", "city", "state")
items = spark.table(f"{catalog}.bronze.order_items") \
    .select("order_item_id", "order_id", "product_id", "quantity", "price")
products = spark.table(f"{catalog}.bronze.products") \
    .select("product_id", "product_name", "category")

fact_sales = items.join(orders, "order_id").join(products, "product_id")
fact_sales.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.{tgt}.fact_sales")

# ---- Revenue by state ----
orders_full = spark.table(f"{catalog}.{src}.orders_enriched")
revenue_by_state = orders_full.groupBy("state") \
    .agg(sum("total_amount").alias("total_revenue"), count("order_id").alias("total_orders"))
revenue_by_state.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.{tgt}.agg_revenue_by_state")

# ---- Top products ----
top_products = fact_sales.groupBy("product_name","category") \
    .agg(sum(col("quantity") * col("price")).alias("revenue")) \
    .orderBy(col("revenue").desc())
top_products.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{catalog}.{tgt}.agg_top_products")

print("✅ Gold layer complete")